 # import

In [1]:
import os, csv, gc, json, random, re, math, shutil, datetime, time, glob

# Turn off all warnings
# 경고 off
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['PYTHONWARNINGS'] = 'ignore'
hf_logging.set_verbosity_error()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


import inspect

import numpy as np
import pandas as pd

from sklearn.metrics import log_loss   
from transformers.trainer_pt_utils import LengthGroupedSampler 
from transformers import (
    DebertaV2Model, TrainingArguments, Trainer, TrainerCallback, AutoTokenizer, EarlyStoppingCallback
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from safetensors.torch import load_file

In [2]:
# GPU Configuration for AWS L40S
# GPU 설정(AWS L40S)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('device:', DEVICE)
print('torch version :', torch.__version__)
print('cuda build    :', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('device count  :', torch.cuda.device_count())

if torch.cuda.is_available():
    print('device name   :', torch.cuda.get_device_name(0))
    print('capability    :', torch.cuda.get_device_capability(0))
    # Cuda test
    # Cuda 연산 테스트
    try:
        x = torch.zeros(3, device='cuda')
        y = x + 1
        print('GPU calculation is available', y.tolist())
    except Exception as e:
        print('GPU calculation is unavailable', e)
else:
    print('No Cuda detected!')

device: cuda
torch version : 2.12.1+cu130
cuda build    : 13.0
cuda available: True
device count  : 1
device name   : NVIDIA L4
capability    : (8, 9)
GPU calculation is available [1.0, 1.0, 1.0]


### 시드 설정

In [3]:
# It must be 42, since it is the ultimate answer to life, the universe and everything
SEED = 42
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f'Seed = {seed}')

set_seeds(42)

Seed = 42


### 데이터셋 불러오기

In [4]:
# load dataset
DATA_DIR = 'origin'

train_df   = pd.read_csv(os.path.join(DATA_DIR, 'train.csv')) # 46729 rows
valid_df   = pd.read_csv(os.path.join(DATA_DIR, 'valid.csv')) # 5000 rows, for model selection
holdout_df = pd.read_csv(os.path.join(DATA_DIR, 'holdout.csv')) # 5748 rows, for final evaluation & boosting

print('train:', train_df.shape)
print('valid:', valid_df.shape)
print('holdout:', holdout_df.shape)

train: (46729, 9)
valid: (5000, 9)
holdout: (5748, 9)


# 데이터셋 전처리

In [5]:
# preprocessing dataset
for d in (train_df, valid_df, holdout_df):
    # 정수 인코딩
    # Preprocessing Dataset, Integer label encoding: 0 = A wins, 1 = B wins, 2 = tie
    d['label'] = np.select(
        [d['winner_model_a'] == 1,
         d['winner_model_b'] == 1,
         d['winner_tie'] == 1],
        [0, 1, 2],
        default= -1,  # reserved for invalid or missing labels
    )

In [6]:
for name, d in zip(('train','valid','holdout'), (train_df, valid_df, holdout_df)):
    target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
    # 매핑 안 된 행(-1) 없나
    # Ensure no rows fall back
    assert (d['label'] != -1).all(), 'There are invalid or missing labels here!'
    
    # 모든 행은 (1, 0, 0), (0, 1, 0), (0, 0, 1) 중 하나여야 한다.
    # Ensure every row is one of these : (1, 0, 0), (0, 1, 0), (0, 0, 1)
    row_sum = d[target_cols].sum(axis=1)
    assert (row_sum == 1).all(), f'There are {int((row_sum != 1).sum())} invalid rows!'
    
    print('*** Validation passed! ***')
    print(f"Label distribution for {name}: {d['label'].value_counts().sort_index().to_dict()}", end = "")
    
    p = d['label'].value_counts(normalize=True).sort_index() * 100
    print(f"(a={p[0]:.2f}%, b={p[1]:.2f}%, tie={p[2]:.2f}%) \n")

*** Validation passed! ***
Label distribution for train: {0: 16312, 1: 15977, 2: 14440}(a=34.91%, b=34.19%, tie=30.90%) 

*** Validation passed! ***
Label distribution for valid: {0: 1745, 1: 1710, 2: 1545}(a=34.90%, b=34.20%, tie=30.90%) 

*** Validation passed! ***
Label distribution for holdout: {0: 2007, 1: 1965, 2: 1776}(a=34.92%, b=34.19%, tie=30.90%) 



### 토크나이징

In [7]:
# Tokenizing
MODEL_NAME = 'OpenAssistant/reward-model-deberta-v3-large-v2'
REVISION   = 'c355404efa9ad2ad069f3a197cae0523c14244fc'   # pinned commit of the official repository / 공식 저장소의 고정 커밋

MAX_LEN   = 2048     
Q_FLOOR   = 256    

# 잘릴 때 앞 60% + 뒤 40%
# if sequences are truncated, get 60% from front and 40% from rear to preserve context
HEAD_FRAC = 0.6       

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
CLS, SEP, PAD = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id

# 입력 구조 :  [CLS] Q [SEP] ANSWER_A [SEP] / [CLS] Q [SEP] ANSWER_B [SEP]
# input sequences : [CLS] Q [SEP] ANSWER_A [SEP] / [CLS] Q [SEP] ANSWER_B [SEP]
BUDGET = MAX_LEN - 3          

### 개행 문자 보존

전처리 시 개행 문자의 빈도 수가 사용자의 선택에 상당한 영향을 끼쳤기에, 토크나이저가 삭제하는 개행 문자를 [BR] 토큰으로 치환하여 보존한다.
인코더가 모델 내부에서 BR 토큰이 log_loss에 미치는 영향을 학습할 수 있다,

In [8]:
# replace \n(line break) to [BR] so that the tokenizer does not erase it (since it is the primary feature)
# so that the model can learn [BR]'s effect directly.
tokenizer.add_tokens(['[BR]'])
BR_ID = tokenizer.convert_tokens_to_ids('[BR]')

# verify
# 토크나이저가 개행을 어떻게 처리하는지 확인
_raw = 'first line\nsecond line\n\nnew paragraph'

def to_model_text(t):
    return t.replace('\n', ' [BR] ')
    
print('Before :', tokenizer.convert_ids_to_tokens(
    tokenizer(_raw, add_special_tokens=False)['input_ids']))
print('After preservation :', tokenizer.convert_ids_to_tokens(tokenizer(_raw.replace('\n', ' [BR] '), add_special_tokens=False)['input_ids']))

Before : ['▁first', '▁line', '▁second', '▁line', '▁new', '▁paragraph']
After preservation : ['▁first', '▁line', '▁', '[BR]', '▁second', '▁line', '▁', '[BR]', '▁', '[BR]', '▁new', '▁paragraph']


In [9]:
def parse_text(x):
    if pd.isna(x):
        return ''
    try:
        turns = json.loads(x)
    except (json.JSONDecodeError, TypeError):
        return str(x)
    if isinstance(turns, list):
        return ' '.join(t for t in turns if isinstance(t, str))
    return str(turns)

# If truncated, preserve k tokens for Question then extract 60% from front, 40% from rear for every Answer
# k개만 남기되 앞 60% + 뒤 40% 추출
def head_tail(ids, k):
    ids = list(ids)                 # implicit casting for unattended dataframe input
    if len(ids) <= k:
        return ids
    h = int(k * HEAD_FRAC)
    t = k - h
    return ids[:h] + ids[-t:] if t > 0 else ids[:h]

# 답변이 안 쓴 예산은 Q가 회수.
# allocate leftover budget for Question
def alloc_q(lq, l_resp):
    kq = min(lq, Q_FLOOR)
    k_resp = min(l_resp, BUDGET - kq)
    slack = BUDGET - kq - k_resp
    return min(lq, kq + slack)

#  길이가 다른 입력에서 동일한 질문 입력 보장
# ensures both branches keep the identical question prefix even when answers of different lengths are truncated
def build_row(iq, ia, ib):
    lq, la, lb = len(iq), len(ia), len(ib)
    kq = alloc_q(lq, max(la, lb))         
    ka = min(la, BUDGET - kq)
    kb = min(lb, BUDGET - kq)
    q = head_tail(iq, kq)
    sa = [CLS] + q + [SEP] + head_tail(ia, ka) + [SEP]
    sb = [CLS] + q + [SEP] + head_tail(ib, kb) + [SEP]
    return sa, sb

### 전처리 객체

In [10]:
# 전처리 객체, 학습 전에 전부 돌려놔서 메모리에 상주시키면 학습 시 CPU 오버헤드가 일어나지 않는다
# Tokenize everything up front and keep it in memory to avoid CPU overhead during training (needs ~100 MB of RAM)
class PrefDataset(Dataset):
    def __init__(self, df, has_label=True, swap_aug=False):
        df = df.reset_index(drop=True)
        self.has_label = has_label
        self.swap_aug = swap_aug          # 학습셋만 True (§2)

        q_raw = df['prompt'].map(parse_text).tolist()
        a_raw = df['response_a'].map(parse_text).tolist()
        b_raw = df['response_b'].map(parse_text).tolist()

        iq = tokenizer([to_model_text(t) for t in q_raw], add_special_tokens=False)['input_ids']
        ia = tokenizer([to_model_text(t) for t in a_raw], add_special_tokens=False)['input_ids']
        ib = tokenizer([to_model_text(t) for t in b_raw], add_special_tokens=False)['input_ids']

        self.seq_a, self.seq_b = [], []
        for x, y, z in zip(iq, ia, ib):
            sa, sb = build_row(x, y, z)
            self.seq_a.append(np.asarray(sa, dtype=np.int32))
            self.seq_b.append(np.asarray(sb, dtype=np.int32))

        # for length sorted batching
        # 길이 정렬 배칭 키 = max(La, Lb) — 스왑 불변
        self.lengths = [max(len(x), len(y)) for x, y in zip(self.seq_a, self.seq_b)]
        self.labels = df['label'].tolist() if has_label else None

    def __len__(self):
        return len(self.seq_a)

    def __getitem__(self, i):
        a, b = self.seq_a[i], self.seq_b[i]
        lab = self.labels[i] if self.has_label else None
        # Swap augment - With a 50% probability, the positions and labels of A and B are swapped.
        # it is harmless because this dataset barely has position bias(34.91% vs 34.19%)
        # 위치 편향이 거의 없는 데이터셋이므로, 50% 확률로 A와 B의 위치를 바꿔서 데이터 증강 효과를 준다.
        if self.swap_aug and random.random() < 0.5:
            a, b = b, a
            if lab is not None and lab != 2:     # tie(2)는 불변
                lab = 1 - lab
        item = {'a_ids': a, 'b_ids': b}
        if lab is not None:
            item['labels'] = lab
        return item

In [11]:
# 동적 패딩, 0으로 패딩된 부분은 학습 시 어텐션 연산의 대상이 되지 않음.
# dynamic padding, padding parts are not multiplied during learning process. 
def collate_fn(batch):
    out = {}
    for key, mkey in [('a_ids', 'a_mask'), ('b_ids', 'b_mask')]:
        seqs = [b[key] for b in batch]
        L = max(len(s) for s in seqs)
        ids  = np.full((len(seqs), L), PAD, dtype=np.int64)
        mask = np.zeros((len(seqs), L), dtype=np.int64)
        for i, s in enumerate(seqs):
            ids[i, :len(s)] = s
            mask[i, :len(s)] = 1
        out[key]  = torch.from_numpy(ids)
        out[mkey] = torch.from_numpy(mask)
    if 'labels' in batch[0]:
        out['labels'] = torch.tensor([b['labels'] for b in batch], dtype=torch.long)
    return out

# 모델 정의

### Attention pooling (scorer initialized N(0, 0.02))

N(0, 0.02)로 초기화된 attention pooling, 특정 토큰에 가중치를 주어 pooling시 오차가 감소한다면 가중치를 늘린다.

In [12]:
# Attention pooling — scorer initialized N(0, 0.02) so gradients flow from the first step.
class AttnPool(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.score = nn.Linear(h, 1)
        nn.init.normal_(self.score.weight, std=0.02) # initalize as mean = 0, std = 0.02
        nn.init.zeros_(self.score.bias)
        self.force_uniform = False        # True → mean pooling 
        
    def forward(self, last_hidden, mask):
        s = self.score(last_hidden).squeeze(-1)              
        if self.force_uniform:
            s = torch.zeros_like(s)
        s = s.float().masked_fill(mask == 0, float('-inf'))  
        w = torch.softmax(s, dim=1).unsqueeze(-1)            
        return (last_hidden.float() * w).sum(dim=1)

# 모델 생성자 — 순수 텍스트 모델 (산술 피처는 사후 스태킹 단계로 이동)
# model ctr
class PrefModel(nn.Module):
    def __init__(self, model_name, num_classes=3,
                 vocab_size=None, new_token_ids=()):
        super().__init__()
        self.encoder, info = DebertaV2Model.from_pretrained(
            model_name, revision=REVISION, dtype=torch.float32, output_loading_info=True)      # implicit fp32 casting
        assert not info['missing_keys'], f"There are no weights loaded!: {info['missing_keys'][:5]}"
        print(f"Removed key:{info['unexpected_keys']}")   # pooler/classifier 만 나와야 정상
        print('Initialized.')

        # [BR] 토큰 등록
        # Register [BR] token
        if new_token_ids:
            emb = self.encoder.get_input_embeddings()
            if vocab_size is not None and vocab_size > emb.weight.size(0):
                self.encoder.resize_token_embeddings(vocab_size)
                emb = self.encoder.get_input_embeddings()
            with torch.no_grad():
                mean_vec = emb.weight[:min(new_token_ids)].mean(0)  # 실제 사전학습된 행들의 평균 / mean of the pretrained embedding rows)
                for tid in new_token_ids:
                    emb.weight[tid] = mean_vec

        h = self.encoder.config.hidden_size       # DeBerta_large = 1024
        self.pool = AttnPool(h)                   # Apply attention pooling
        self.LN = nn.LayerNorm(h)                 # Shared LayerNorm for both pooled vectors (one module keeps swap symmetry)
        in_dim = h * 4                            # {[a, b, a - b, tanh(a) * tanh(b)] | a =[CLS] Q [SEP] ans_a [SEP] , b = [CLS] Q [SEP] ans_b [SEP] } = 4096
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(in_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.1), 
            nn.Linear(1024, num_classes),
        )
        self.loss_fct = nn.CrossEntropyLoss()

    def gradient_checkpointing_enable(self, **kwargs):
        self.encoder.gradient_checkpointing_enable(**kwargs)

    def _encode(self, ids, mask):
        out = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        return self.pool(out, mask)

    def forward(self, a_ids, a_mask, b_ids, b_mask, labels=None):
            # training at AMP_DTYPE(bf16)
            # 학습은 AMP_DTYPE(bf16)으로
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=torch.cuda.is_available()): # for supporting multi-precision between training pods and kaggle 
            a = self._encode(a_ids, a_mask)
            b = self._encode(b_ids, b_mask)
            emb_a = self.LN(a)
            emb_b = self.LN(b)
            x = torch.cat([emb_a, emb_b, emb_a - emb_b, torch.tanh(emb_a) * torch.tanh(emb_b)], dim=1) # [a, b, a - b, tanh(a) * tanh(b)]
            logits = self.head(x)

        logits = logits.float()                       # 손실/지표 fp32 로 // loss/indicator at fp32
        
        if labels is None:
            return {'logits': logits}                
        return {'loss': self.loss_fct(logits, labels), 'logits': logits}

# 학습

### OOM 방지!

In [13]:
# remove any unused variables, basically remove trainer, model class and extra
# 기본으로 trainer/model 을 지우고, extra 로 넘긴 변수명도 함께 삭제
def mem_free(*extra):
    for g in ['trainer', 'model', *extra]:
        if g in globals():
            del globals()[g]       
    # to break circular references
    # PyTorch 모델은 순환참조가 있어 여러 번 돌려 확실히 회수
    for _ in range(3):
        gc.collect() # 참조를 끊은 후 회수해야 메모리 반환 가능
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print('clear!')

mem_free()

clear!


### 하이퍼파라미터 설정

In [14]:
# Hyperparameter setup
LR           = 5e-6
EPOCHS       = 3

WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRADIENT_CLIPPING = 1.0

BATCH_SIZE   = 8
GRADIENT_ACCUMULATION_STEPS = 2 # effective batch = 8 * 2 = 16

# Use bf16(for L40S)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

OUTPUT_DIR  = '../outputs'
LOG_CSV_DIR = '../outputs'
USE_COMPILE = False 

training_args = TrainingArguments(         
    output_dir=f'{OUTPUT_DIR}/run',         
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=16,          
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=GRADIENT_CLIPPING, 
    
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_logloss',
    greater_is_better=False,
    
    bf16=USE_BF16,
    fp16=not USE_BF16 and torch.cuda.is_available(),  
    gradient_checkpointing=True,            
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=4,
    
    # keep optimizer/scheduler/RNG state so training can resume
    save_only_model=False,                  
    torch_compile=USE_COMPILE,
    
    eval_accumulation_steps=10,
    logging_steps=50,
    report_to='none',
    disable_tqdm=False,  
    remove_unused_columns=False,
    seed=SEED,
)

tr, va, ho = train_df, valid_df, holdout_df

tr = tr.reset_index(drop=True)
va = va.reset_index(drop=True)
ho = ho.reset_index(drop=True)

steps_per_epoch = math.ceil(len(tr) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print(f'Training {len(tr):,} rows / valid set {len(va):,} rows / hold-out {len(ho):,} rows')
p = va['label'].value_counts(normalize=True).sort_index() * 100

print(f"""
==================================================
[Dataset]
- Training set  : {len(tr):,} rows
- Valid set     : {len(va):,} rows , log_loss SE ≈ ±{0.6 / len(va) ** 0.5:.4f}
- Valid ratio   : a={p[0]:.2f}% / b={p[1]:.2f}% / tie={p[2]:.2f}%
- Hold-out set  : {len(ho):,} rows

[Optimization]
- EPOCHS                      : {training_args.num_train_epochs}
- BATCH_SIZE (per device)     : {training_args.per_device_train_batch_size}
- GRADIENT_ACCUMULATION_STEPS : {training_args.gradient_accumulation_steps}
- Effective Batch Size        : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}
- LEARNING_RATE               : {training_args.learning_rate}
- WEIGHT_DECAY                : {training_args.weight_decay}
- WARMUP_RATIO                : {training_args.warmup_ratio}  (≈ {warmup_steps:,} / {total_steps:,} steps)
- MAX_GRAD_NORM               : {training_args.max_grad_norm}
- MAX_LEN                     : {MAX_LEN}
- SEED                        : {SEED}

[Eval & Checkpoint]
- eval/save Frequency    : steps, every {training_args.eval_steps} (≈ {training_args.eval_steps / steps_per_epoch:.2f} epoch)
- steps/epoch · total    : {steps_per_epoch:,} · {total_steps:,}
- save_total_limit       : {training_args.save_total_limit}
- Model Selection        : {training_args.metric_for_best_model} (greater_is_better={training_args.greater_is_better}), load_best_model_at_end={training_args.load_best_model_at_end}
- per_device_eval_batch  : {training_args.per_device_eval_batch_size}
- eval_accumulation      : {training_args.eval_accumulation_steps}

[System & Precision]
- Compute dtype (AMP)    : {AMP_DTYPE}
- fp16 / bf16 (args)     : {training_args.fp16} / {training_args.bf16}
- GradScaler             : {'on (fp16)' if training_args.fp16 else 'off (bf16/fp32)'}
- Gradient Checkpointing : {training_args.gradient_checkpointing}
- Torch Compile          : {USE_COMPILE}
- save_only_model        : {training_args.save_only_model}
- dataloader workers     : {training_args.dataloader_num_workers}
- Output Directory       : {OUTPUT_DIR}
==================================================
""")

Training 46,729 rows / valid set 5,000 rows / hold-out 5,748 rows

[Dataset]
- Training set  : 46,729 rows
- Valid set     : 5,000 rows , log_loss SE ≈ ±0.0085
- Valid ratio   : a=34.90% / b=34.20% / tie=30.90%
- Hold-out set  : 5,748 rows

[Optimization]
- EPOCHS                      : 3
- BATCH_SIZE (per device)     : 8
- GRADIENT_ACCUMULATION_STEPS : 2
- Effective Batch Size        : 16
- LEARNING_RATE               : 5e-06
- WEIGHT_DECAY                : 0.01
- WARMUP_RATIO                : 0.1  (≈ 876 / 8,763 steps)
- MAX_GRAD_NORM               : 1.0
- MAX_LEN                     : 2048
- SEED                        : 42

[Eval & Checkpoint]
- eval/save Frequency    : steps, every 500 (≈ 0.17 epoch)
- steps/epoch · total    : 2,921 · 8,763
- save_total_limit       : 1
- Model Selection        : eval_logloss (greater_is_better=False), load_best_model_at_end=True
- per_device_eval_batch  : 16
- eval_accumulation      : 10

[System & Precision]
- Compute dtype (AMP)    : torch.bfloa

## 헬퍼 함수 정의 / Helper function Def.

### TrainingLogger - logging every steps for visualization

In [15]:
# Custom CSV logger called in every evalution logic, basically supporting multiple gpus with is_process_zero
# 채점 시마다 호출되어 학습률, 평가 결과 등을 저장하는 CSV 로거 콜백, 다중 GPU의 경우 첫 번째 GPU에서만 기록함.
class TrainingLogger(TrainerCallback):
    def __init__(self, filepath):
        self.filepath = filepath
        self.fieldnames = [
            'step', 'epoch', 'loss', 'learning_rate',
            'eval_loss', 'eval_logloss', 'eval_accuracy',
        ]
        self._seen = set()   # for dataparllel

    def on_train_begin(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            self._seen.clear()
            os.makedirs(os.path.dirname(self.filepath), exist_ok=True)
            # 파일이 비어있으면 헤더 작성
            if not os.path.exists(self.filepath) or os.path.getsize(self.filepath) == 0:
                with open(self.filepath, 'a', newline='', encoding='utf-8') as f:
                    csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction='ignore').writeheader()

    # logging protocol
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not (state.is_world_process_zero and logs is not None):
            return
        # 로그 종류 판별: eval 로그 / train 로그 / 그 외(skip)
        if 'eval_loss' in logs:
            kind = 'eval'
        elif 'loss' in logs:
            kind = 'train'
        else:
            return
        # 같은 step의 같은 종류 로그가 또 오면(Trainer 중복 방출) 건너뜀
        key = (state.global_step, kind)
        if key in self._seen:
            return
        self._seen.add(key)

        row = {k: '' for k in self.fieldnames}
        row['step'] = state.global_step
        row['epoch'] = logs.get('epoch', state.epoch)
        for k in self.fieldnames:
            if k in logs:
                row[k] = logs[k]
        with open(self.filepath, 'a', newline='', encoding='utf-8') as f:
            csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction='ignore').writerow(row)


# we will record log_loss and accuracy for this logger
#  지표 : log_loss + accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(np.array(logits), dtype=torch.float32)
    probs = torch.softmax(logits, dim=1).numpy()
    preds = probs.argmax(axis=1)
    return {
        'logloss':  log_loss(labels, probs, labels=[0, 1, 2]),
        'accuracy': (preds == labels).mean(),
    }

### BestSavor - preserving intermediate good ones

In [16]:
# BestSavor - 중간 단계 체크포인트 저장용 콜백

# Trainer's save_total_limit keeps only the MOST RECENT checkpoints (sorted by step, not by score) — the best one is protected but mid-run good ones get deleted.
# This callback copies the top-K scorers into a separate folder so checkpoint ensembling stays possible. Weights only: optimizer/scheduler/RNG are skipped.
# require large storage but saves intermediate good ones
# Trainer의 save_total_limit은 '스텝 순 최신' 기준이라 중간 구간의 좋은 체크포인트가 지워진다. 이 콜백은 점수 상위 K개를 별도 폴더에 복사해 앙상블 재료를 남긴다.
class BestSaver(TrainerCallback):
    def __init__(self, k=5, output_dir=None, metric='eval_logloss', greater_is_better=False):
        self.k = k
        self.metric = metric                          # must match metric_for_best_model
        self.sign = -1 if greater_is_better else 1    # sort ascending on sign * score
        self.skip = shutil.ignore_patterns('optimizer.pt', 'scheduler.pt',
                                           'rng_state*.pth', 'scaler.pt')
        self.best_dir = os.path.join(output_dir, 'best_checkpoints')
        self.state_file = os.path.join(self.best_dir, 'best_k_state.json')
        self.best_checkpoints = []
        os.makedirs(self.best_dir, exist_ok=True)
        self._load_state()

    def _load_state(self):
        # Restore the ranking so a resumed run does not start counting over
        # 재개 시 랭킹 복구 — 처음부터 다시 세지 않도록
        if os.path.exists(self.state_file):
            try:
                with open(self.state_file, 'r', encoding='utf-8') as f:
                    self.best_checkpoints = json.load(f)
                print(f'[BestSaver] state restored ({len(self.best_checkpoints)} entries)')
            except Exception as e:
                print(f'[BestSaver] state load failed, starting fresh: {e}')

    def _save_state(self):
        for rank, entry in enumerate(self.best_checkpoints, 1):
            entry['rank'] = rank
        try:
            with open(self.state_file, 'w', encoding='utf-8') as f:
                json.dump(self.best_checkpoints, f, indent=4, ensure_ascii=False)
        except Exception as e:
            print(f'[BestSaver] state save failed: {e}')

    def on_save(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return

        src = os.path.join(args.output_dir, f'checkpoint-{state.global_step}')
        if not os.path.isdir(src):
            raise FileNotFoundError(f'[BestSaver] checkpoint not found: {src}')

        # Look up this step's score. With eval_steps == save_steps the Trainer evaluates
        # before it saves, so the entry is already in log_history.
        # 이번 스텝의 점수 조회 — eval_steps == save_steps면 평가가 저장보다 먼저 실행된다.
        score = None
        for log in reversed(state.log_history):
            if log.get('step') == state.global_step and self.metric in log:
                score = log[self.metric]
                break
        if score is None:                                    # fallback / 폴백
            for log in reversed(state.log_history):
                if log.get('step') == state.global_step and 'eval_loss' in log:
                    score = log['eval_loss']
                    break
        if score is None:
            # Warn rather than raise: losing a 20-hour run to a logging hiccup is not worth it
            # 예외 대신 경고 — 로깅 타이밍 문제로 20시간짜리 런을 잃을 이유는 없다
            print(f'[BestSaver] no eval score at step {state.global_step}, skipped')
            return

        entry = {
            'score': score,
            'metric': self.metric,
            'step': state.global_step,
            'epoch': round(state.epoch, 4) if state.epoch else 0,
            'saved_at': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        self.best_checkpoints.append(entry)
        self.best_checkpoints.sort(key=lambda x: self.sign * x['score'])
        self._save_state()

        # Copy in if this step made the top-K / top-K에 들면 복사
        if any(e['step'] == state.global_step for e in self.best_checkpoints[:self.k]):
            dest = os.path.join(self.best_dir, f'checkpoint-{state.global_step}')
            if not os.path.exists(dest):
                rank = self.best_checkpoints.index(entry) + 1
                print(f'[BestSaver] rank {rank} ({self.metric}={score:.4f}) -> {dest}')
                for i in range(3):          # retry — the copy can race with Trainer's rotation
                    try:
                        shutil.copytree(src, dest, ignore=self.skip)
                        break
                    except Exception as e:
                        shutil.rmtree(dest, ignore_errors=True)
                        if i == 2:
                            print(f'[BestSaver] copy failed after retries: {e}')
                            raise
                        time.sleep(3)

        # Delete whatever was pushed out of the top-K / 상위 K에서 밀려난 사본 삭제
        if len(self.best_checkpoints) > self.k:
            dropped = self.best_checkpoints[self.k:]
            self.best_checkpoints = self.best_checkpoints[:self.k]
            self._save_state()
            for e in dropped:
                path = os.path.join(self.best_dir, f'checkpoint-{e["step"]}')
                if os.path.exists(path):
                    print(f'[BestSaver] removing {e["score"]:.4f} @ step {e["step"]}')
                    shutil.rmtree(path, ignore_errors=True)

### 길이 정렬 Batch를 위해 custom trainer override

In [17]:
# Custom samplers — length-grouped batches for training, Length-Sorted Batches for eval (kills padding waste)
# 커스텀 샘플러 — 학습은 길이 그룹 배칭, 평가는 길이 정렬 배칭 (패딩 낭비 제거: eval 689s → ~200s)
class GroupedTrainer(Trainer):
    def _get_train_sampler(self, train_dataset=None):
        ds = train_dataset if train_dataset is not None else self.train_dataset
        return LengthGroupedSampler(
            batch_size=self.args.train_batch_size * self.args.gradient_accumulation_steps,
            dataset=ds, lengths=ds.lengths, model_input_name=None,
        )

    def _get_eval_sampler(self, eval_dataset):
        return np.argsort(np.asarray(eval_dataset.lengths)).tolist()

# if the parent signature differs, the override is silently ignored and eval stays slow
# 부모 시그니처가 다르면 LSB override가 조용히 무시된다.
for name in ['_get_train_sampler', '_get_eval_sampler']:
    base, mine = inspect.signature(getattr(Trainer, name)), inspect.signature(getattr(GroupedTrainer, name))
    assert list(base.parameters) == list(mine.parameters), f'[DEBUG] {name} signature mismatch — LSB override not applied!'
print(f'*** LSB activated! ***')

*** LSB activated! ***


### 데이터셋 불러오기

In [18]:
# Load dataset, Enable Swap Augment only on training set
# train set에서만 Swap 증강 활성화
train_ds   = PrefDataset(tr, swap_aug=True)  
valid_ds   = PrefDataset(va)
holdout_ds = PrefDataset(ho)

os.makedirs(OUTPUT_DIR, exist_ok=True)      

### Smoke test

In [19]:
# 0) 모델, 트레이너 객체 생성
# 0) Construct model, trainer
model = PrefModel(MODEL_NAME, vocab_size=len(tokenizer),
                  new_token_ids=[BR_ID]).to(DEVICE)   

trainer = GroupedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[
        TrainingLogger(f'{LOG_CSV_DIR}/train_log.csv')
    ],
)
# 스모크 테스트 본학습 전 필수 점검 

# 1) 전처리 불변식 3,000행 전수: 두 브랜치의 Q 프리픽스 동일 + 길이 한도 준수
# 1) Preprocessing invariant over 3,000 rows: both branches share the identical question prefix, and neither branch exceeds MAX_LEN.
for sa, sb in zip(train_ds.seq_a[:3000], train_ds.seq_b[:3000]):
    qa = list(sa[:list(sa).index(SEP)])
    qb = list(sb[:list(sb).index(SEP)])
    assert qa == qb, 'Asymmetric question prefix between the a/b branches!'
    assert len(sa) <= MAX_LEN and len(sb) <= MAX_LEN, 'Sequence longer than MAX_LEN!'
    
# 2) Trainer 경유 라벨 없는 predict — 헤드·콜레이터 배선 버그를 잡는다
# 2) Label-free predict through the Trainer — catches head/collator wiring bugs before the long run.
_smoke = PrefDataset(tr.iloc[:4], has_label=False)
_ = trainer.predict(_smoke)
print('Smoke tests 1 & 2 passed')

# 3) 최악배치 프로브: 최장 행 BATCH*GA개로 '실제 Trainer 경로' 1 옵티마이저 스텝.
#    fwd/bwd + gradient checkpointing + AdamW 상태 할당(피크 VRAM의 큰 몫)까지 본학습과 동일 경로로 검증, 여기서 OOM이면 본학습도 난다.

# 3) Worst-batch probe: one real optimizer step on the longest BATCH*GA rows through the actual
#    Trainer path (fwd/bwd + gradient checkpointing + AdamW state allocation, which dominates peak VRAM).
#    A hand-rolled probe would miss those; if this OOMs, the real run will too -> lower BATCH_SIZE/raise GA.
#    Also the first VRAM check of the new 4-way head (in_dim = 4096).

_idx = np.argsort(train_ds.lengths)[-(BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS):]
print('Probe batch lengths:', sorted(train_ds.lengths[i] for i in _idx))

class _Probe(Dataset):
    def __init__(self, base, idx): self.base, self.idx = base, list(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return self.base[self.idx[i]]
    @property
    def lengths(self): return [self.base.lengths[i] for i in self.idx]

_snap = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}   # weight snapshot (~1.7 GB RAM)
_pa = TrainingArguments(                       # memory-path settings must mirror the real run
    output_dir=f'{OUTPUT_DIR}/_probe',
    max_steps=1,                               # GA micro-batches + a single optimizer step
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LR, weight_decay=WEIGHT_DECAY,
    max_grad_norm=GRADIENT_CLIPPING,
    bf16=USE_BF16,                             # must match the real run — fp16 and bf16 differ in
    fp16=not USE_BF16 and torch.cuda.is_available(),   # memory path (GradScaler) / 본학습과 동일해야 유효한 검증
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2,
    report_to='none', disable_tqdm=False, remove_unused_columns=False,
    seed=SEED,
)
torch.cuda.reset_peak_memory_stats()
try:
    _pt = GroupedTrainer(model=model, args=_pa,
                         train_dataset=_Probe(train_ds, _idx), data_collator=collate_fn)
    _pt.train()
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f'Worst-batch probe passed — peak VRAM {peak:.1f} GiB')

    # 3b) Eval-path worst batch. The real run evaluates with per_device_eval_batch_size=16 and
    #     length-sorted batching, so its last eval batch is 16 x 2048 — the largest attention
    #     matrix this run ever builds. Measured here, while the optimizer state is still resident,
    #     because that is what the first real eval (step 750) also sees.
    #     Must go through `trainer` (eval batch 16); `_pt` would use 8 (`_pa` sets no eval batch).
    # 3b) 평가 경로 최악 배치. 본학습 eval 은 배치 16 + 길이 정렬이라 마지막 배치가 16 x 2048 이고,
    #     이게 이 런에서 만들어지는 가장 큰 어텐션 행렬이다. 옵티마이저 상태가 아직 상주하는 지금
    #     재야 본학습 첫 eval(step 750)과 같은 조건이 된다.
    #     반드시 `trainer`(eval 배치 16)로 부른다 — `_pt` 는 `_pa` 에 eval 배치가 없어 8 이 된다.
    torch.cuda.reset_peak_memory_stats()
    _ = trainer.predict(_Probe(train_ds, _idx))
    print(f'Eval-path probe passed  — peak VRAM '
          f'{torch.cuda.max_memory_allocated() / 2**30:.1f} GiB')
finally:
    # Always restore weights and free the snapshot, even if the probe OOMs mid-step.
    # 프로브가 도중에 죽더라도 가중치 복원과 스냅샷 해제는 반드시 수행한다.
    model.load_state_dict(_snap)               # undo the probe's single optimizer step
    del _snap
    for name in ['_pt', '_pa']:
        if name in globals():
            del globals()[name]
    for _ in range(3):
        gc.collect()
    torch.cuda.empty_cache()

print('All smoke tests passed — safe to proceed')

Removed key:['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
Initialized.


Smoke tests 1 & 2 passed
Probe batch lengths: [2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048]


Step,Training Loss


Worst-batch probe passed — peak VRAM 16.0 GiB


Eval-path probe passed  — peak VRAM 21.3 GiB


All smoke tests passed — safe to proceed


In [20]:
mem_free()

clear!


### Training Loop

In [21]:
model = PrefModel(MODEL_NAME, vocab_size=len(tokenizer),
                  new_token_ids=[BR_ID]).to(DEVICE)   

trainer = GroupedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[TrainingLogger(f'{LOG_CSV_DIR}/train_log.csv'),
               BestSaver(k=7, output_dir=OUTPUT_DIR, metric='eval_logloss', greater_is_better=False),
               EarlyStoppingCallback(early_stopping_patience=5)],
)

Removed key:['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
Initialized.


In [22]:
# TTA: swap the (a, b) slots, predict again, and average the two views in logit space
# TTA: A/B 자리를 바꾼 뷰로 한 번 더 예측해 로짓 공간에서 평균
class SwappedDataset(Dataset):
    def __init__(self, base):
        self.base = base
        self.lengths = base.lengths          # eval sampler sorts by this / 길이 정렬 샘플러가 참조
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        b = self.base
        return {'a_ids': b.seq_b[i], 'b_ids': b.seq_a[i]}

def unsort(pred, lengths):
    # predict() returns rows in the eval sampler's order — restore original row order, otherwise logits no longer line up with ds.labels (silent corruption).
    # 평가 샘플러 순서로 나온 예측을 원래 행 순서로 복원 — 없으면 라벨과 어긋난 채 조용히 오염된다.
    order = np.argsort(np.asarray(lengths))  
    out = np.empty_like(pred)
    out[order] = pred
    return out

# 모델의 최종 로짓을 저장 / save model's logits for boosting & stacking & fitting
def predict_tta_logits(t, ds):
    z_fwd = unsort(t.predict(ds).predictions, ds.lengths)
    z_swp = unsort(t.predict(SwappedDataset(ds)).predictions, ds.lengths)[:, [1, 0, 2]]  # realign a/b logits
    return z_fwd, z_swp, (z_fwd + z_swp) / 2

In [23]:
# Resume-aware training: continue from the last checkpoint in output_dir if one exists
# 중단 시 자동 재개: output_dir에 체크포인트가 있으면 이어서, 없으면 처음부터
from transformers.trainer_utils import get_last_checkpoint
last_ckpt = get_last_checkpoint(training_args.output_dir) \
    if os.path.isdir(training_args.output_dir) else None
print('Resume from:', last_ckpt or 'none — training from scratch')
trainer.train(resume_from_checkpoint=last_ckpt)

Resume from: none — training from scratch


Step,Training Loss,Validation Loss,Logloss,Accuracy
500,1.042800,1.047279,1.047279,0.445000
1000,1.023300,1.038534,1.038534,0.463800
1500,1.028000,1.009993,1.009993,0.491200
2000,1.016900,1.012764,1.012764,0.487400
2500,1.016400,1.004590,1.004590,0.491200
3000,0.926600,1.014159,1.014159,0.500800
3500,0.959700,1.010470,1.010470,0.501600
4000,0.902400,1.025836,1.025836,0.503200
4500,0.920600,1.021516,1.021516,0.499800
5000,0.918000,1.021025,1.021025,0.503400


[BestSaver] rank 1 (eval_logloss=1.0473) -> ../outputs/best_checkpoints/checkpoint-500


[BestSaver] rank 1 (eval_logloss=1.0385) -> ../outputs/best_checkpoints/checkpoint-1000


[BestSaver] rank 1 (eval_logloss=1.0100) -> ../outputs/best_checkpoints/checkpoint-1500


...

[BestSaver] rank 6 (eval_logloss=1.0215) -> ../outputs/best_checkpoints/checkpoint-4500


[BestSaver] removing 1.0385 @ step 1000


[BestSaver] rank 6 (eval_logloss=1.0210) -> ../outputs/best_checkpoints/checkpoint-5000


[BestSaver] removing 1.0258 @ step 4000


TrainOutput(global_step=5000, training_loss=0.9780898338317872, metrics={'train_runtime': 48784.8338, 'train_samples_per_second': 2.874, 'train_steps_per_second': 0.18, 'total_flos': 0.0, 'train_loss': 0.9780898338317872, 'epoch': 1.7117425539198905})

# Save

### 아키텍처 저장

In [24]:
# Save Architecture
# BestSaver copies weights only, architecture and tokenizer are identical across all copies,
# BestSaver 사본은 가중치뿐이고 구조·토크나이저는 전부 동일하므로, 폴더 루트에 공용 1벌만 두면 재사용할 수 있다.(커스텀 nn.Module 이라 Trainer 가 config 를 안 남긴다).

BEST_DIR  = os.path.join(OUTPUT_DIR, 'best_checkpoints')
BEST_STEP = int(re.sub(r'\D', '', os.path.basename(trainer.state.best_model_checkpoint)))

trainer.model.encoder.config.save_pretrained(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)

print(f'config/tokenizer saved at {BEST_DIR}')
print(f'  {sorted(f for f in os.listdir(BEST_DIR) if not f.startswith("checkpoint-"))}')
print(f'  checkpoints: {sorted(int(p.rsplit("-", 1)[1]) for p in glob.glob(BEST_DIR + "/checkpoint-*"))}')
print(f'  trainer best = checkpoint-{BEST_STEP}')

config/tokenizer saved at ../outputs/best_checkpoints
  ['added_tokens.json', 'best_k_state.json', 'config.json', 'special_tokens_map.json', 'spm.model', 'tokenizer.json', 'tokenizer_config.json']
  checkpoints: [1500, 2000, 2500, 3000, 3500, 4500, 5000]
  trainer best = checkpoint-2500


### 로짓 저장 (스태킹·T 피팅·부스팅 재료)

In [25]:
# Export both view logits (material for stacking / T-fitting / boosting) for every saved models.
META = dict(valid_labels=np.array(valid_ds.labels), hold_labels=np.array(holdout_ds.labels),
            valid_ids=va['id'].to_numpy(),          hold_ids=ho['id'].to_numpy())
ll = lambda z, y: log_loss(y, torch.softmax(torch.tensor(z, dtype=torch.float32), 1).numpy(),
                           labels=[0, 1, 2])

for path in sorted(glob.glob(os.path.join(BEST_DIR, 'checkpoint-*')), key=lambda p: int(p.rsplit('-', 1)[1])):
    step = int(path.rsplit('-', 1)[1])
    trainer.model.load_state_dict(load_file(os.path.join(path, 'model.safetensors')))
    vf, vs, vavg = predict_tta_logits(trainer, valid_ds)
    hf, hs, havg = predict_tta_logits(trainer, holdout_ds)
    np.savez(os.path.join(path, 'logits_export.npz'),
             valid_fwd=vf, valid_swp=vs, hold_fwd=hf, hold_swp=hs, **META)
    print(f'  checkpoint-{step:<6} TTA log_loss  valid {ll(vavg, META["valid_labels"]):.4f}  '
          f'hold {ll(havg, META["hold_labels"]):.4f}')

gc.collect()
torch.cuda.empty_cache()

shutil.copy2(os.path.join(BEST_DIR, f'checkpoint-{BEST_STEP}', 'logits_export.npz'),
             f'{OUTPUT_DIR}/logits_export.npz')
print(f'\n{len(glob.glob(BEST_DIR + "/checkpoint-*/logits_export.npz"))} logits exported')

  checkpoint-1500   TTA log_loss  valid 1.0076  hold 0.9973


  checkpoint-2000   TTA log_loss  valid 1.0092  hold 0.9914


  checkpoint-2500   TTA log_loss  valid 1.0014  hold 0.9928


  checkpoint-3000   TTA log_loss  valid 1.0098  hold 1.0058


  checkpoint-3500   TTA log_loss  valid 1.0091  hold 0.9974


  checkpoint-4500   TTA log_loss  valid 1.0212  hold 1.0135


  checkpoint-5000   TTA log_loss  valid 1.0204  hold 1.0113

7 logits exported


# 검증

### T Scaling

In [26]:
# load best checkpoint's logits
_z     = np.load(f'{OUTPUT_DIR}/logits_export.npz')
v_avg  = (_z['valid_fwd'] + _z['valid_swp']) / 2
h_avg  = (_z['hold_fwd']  + _z['hold_swp'])  / 2
vl, hl = _z['valid_labels'], _z['hold_labels']

In [27]:
# Evaluation
# Temperature scaling: fit a single scalar T on the validation TTA logits
# T 스케일링: valid TTA 로짓으로 스칼라 T 하나만 피팅 (확신도만 보정)
def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(labels)
    logT = torch.zeros(1, requires_grad=True)      # log-param keeps T > 0 / T > 0 보장
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(z / logT.exp(), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(logT.exp().detach())

T = fit_temperature(v_avg, vl)
print(f'T = {T:.4f}   (fitted on valid {len(vl):,} rows)')
print(f'T > 1 → model was overconfident. \n probabilities softened!' if T > 1
      else f' T < 1 → model was underconfident. \n probabilities sharpened!')

T = 0.9463   (fitted on valid 5,000 rows)
 T < 1 → model was underconfident. 
 probabilities sharpened!


### 지표 계산 - TTA + T 적용값

In [28]:
# hold-out final score - apply TTA & T scaling
UNIFORM = 1.0986      # when you assign (1/3, 1/3, 1/3) to all(ln 3)

holdout_probs  = torch.softmax(torch.tensor(h_avg / T, dtype=torch.float32), 1).numpy()
holdout_logloss = log_loss(hl, holdout_probs, labels=[0, 1, 2])

print(f'  hold-out log_loss : {holdout_logloss:.4f}   (n={len(hl):,}, TTA + T={T:.4f})')
print(f'  accuracy          : {(holdout_probs.argmax(1) == hl).mean() * 100:.2f}%')
print(f'  vs UNIFORM {UNIFORM} : {holdout_logloss - UNIFORM:+.4f}')

  hold-out log_loss : 0.9922   (n=5,748, TTA + T=0.9463)
  accuracy          : 50.21%
  vs UNIFORM 1.0986 : -0.1064


### 축 분해 : A/B 이진 logloss로 평가

$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{tie}} + \mathcal{L}_{\text{win}} \cdot (\text{non-tie \%})
$$

손실은 두 개의 이진 질문으로 나뉜다.<br>
$\mathcal{L}_{\text{tie}}$는 "무승부인가?"를,&nbsp;&nbsp;$\mathcal{L}_{\text{win}}$은 "무승부가 아니라면 A가 이기는가?"를 묻는다.

총 logloss는 두 축으로 정확히 분해되므로 각각 따로 잴 수 있다.<br>
아래 코드는 축별 지표(logloss, AUC)를 계산한다.

- **tie 축** — 전체 행에서 $p_{\text{tie}}$로 무승부 여부를 예측 - **전체 행을 Tie/Non-Tie로 이진 분류 했을 때의 logloss**
- **승부 축** — 비-tie 행에서 $p_A / (p_A + p_B)$로 A 승리 여부를 예측 - **해당 행들을 win_A, win_B로 이진 분류 했을 때의 logloss**

마지막은 두 축을 합친 값이 직접 계산한 3-class logloss와 일치하는지 확인한다.

<hr>

### Loss Decomposition: Scoring on the A/B Axis

$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{tie}} + \mathcal{L}_{\text{win}} \cdot (\text{non-tie \%})
$$

The loss splits into two binary questions.<br>
$\mathcal{L}_{\text{tie}}$ asks "is it a tie?", and $\mathcal{L}_{\text{win}}$ asks "if not, does A win?"

The total logloss decomposes exactly into these two axes, so each can be measured on its own.<br>
The code below computes per-axis metrics (logloss, AUC) on the holdout.

- **Tie axis** — predict tie vs non-tie with $p_{\text{tie}}$ over all rows: the logloss of treating the task as a Tie / Non-Tie binary classifier
- **Win axis** — predict whether A wins with $p_A / (p_A + p_B)$ over non-tie rows: the logloss of treating those rows as a win_A / win_B binary classifier

The last line checks that combining the two axes as above reproduces the directly computed 3-class logloss.

In [29]:
def binary(score, lab):
    s = np.clip(score, 1e-12, 1 - 1e-12)
    ll = -(lab * np.log(s) + (1 - lab) * np.log(1 - s)).mean()
    o = np.argsort(s); rank = np.empty(len(s)); rank[o] = np.arange(1, len(s) + 1)
    n1 = lab.sum(); n0 = len(lab) - n1
    auc = (rank[lab.astype(bool)].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    q = lab.mean()
    return ll, auc, -(q * np.log(q) + (1 - q) * np.log(1 - q))

nt = (hl != 2)                                              # 비-tie 마스크
ll_tie, auc_tie, base_tie = binary(holdout_probs[:, 2], (hl == 2).astype(float))
ra = holdout_probs[nt, 0] / (holdout_probs[nt, 0] + holdout_probs[nt, 1])
ll_win, auc_win, base_win = binary(ra, (hl[nt] == 0).astype(float))
frac = nt.mean()

print(f'  tie binary cls result   n={len(hl):6,}  (tie vs non-tie)  logloss {ll_tie:.4f}   AUC {auc_tie:.4f}   baseline {base_tie:.4f}')
print(f'  win binary cls result   n={nt.sum():6,}  (A vs B)          logloss {ll_win:.4f}   AUC {auc_win:.4f}   baseline {base_win:.4f}')
print(f'  verify           tie {ll_tie:.4f} + win {ll_win:.4f} x {frac:.4f} = {ll_tie + ll_win * frac:.6f}'
      f'   vs 3-class direct {holdout_logloss:.6f}   (diff {abs(ll_tie + ll_win * frac - holdout_logloss):.1e})')

  tie binary cls result   n= 5,748  (tie vs non-tie)  logloss 0.5928   AUC 0.6381   baseline 0.6183
  win binary cls result   n= 3,972  (A vs B)          logloss 0.5780   AUC 0.7602   baseline 0.6931
  verify           tie 0.5928 + win 0.5780 x 0.6910 = 0.992170   vs 3-class direct 0.992170   (diff 3.8e-09)


In [30]:
print(f'*** Done. ***')

*** Done. ***


# 끝.